# Evaluating LLMs for Rare Disease Diagnosis: Exploratory Data Analysis

**Portfolio Project** | AI × Healthcare Research  
**Dataset**: Kaggle — *Evaluate LLMs for Rare Disease Diagnosis*  
**Research Goal**: Understand the structure and characteristics of rare disease case data to inform LLM benchmarking strategies

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Consistent style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# Load datasets
df_train = pd.read_csv('train_split_50k_cases.csv')
df_eval  = pd.read_csv('Evaluation.csv')

print(f'Training set : {df_train.shape[0]:,} rows × {df_train.shape[1]} columns')
print(f'Evaluation set: {df_eval.shape[0]:,} rows × {df_eval.shape[1]} columns')
df_train.head(2)

## 2. Dataset Overview & Missing Values

In [ ]:
def missing_summary(df, name):
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(1)
    summary = pd.DataFrame({'Missing': missing, 'Missing %': pct})
    summary = summary[summary['Missing'] > 0].sort_values('Missing %', ascending=False)
    print(f'\n--- {name} ---')
    print(summary.to_string())

missing_summary(df_train, 'Training Set (50k)')
missing_summary(df_eval,  'Evaluation Set (6.9k)')

In [ ]:
# Visualise missingness
cols_of_interest = ['Synonyms', 'ICD-10', 'ICD-11', 'OMIM', 'GARD', 'UMLS']

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (df, name) in zip(axes, [(df_train, 'Training (50k)'), (df_eval, 'Evaluation (6.9k)')]):
    pcts = [df[c].isnull().mean() * 100 for c in cols_of_interest]
    bars = ax.barh(cols_of_interest, pcts, color=sns.color_palette('muted')[1])
    ax.set_xlim(0, 100)
    ax.set_xlabel('% Missing')
    ax.set_title(f'Missing Values — {name}')
    for bar, pct in zip(bars, pcts):
        ax.text(pct + 1, bar.get_y() + bar.get_height()/2, f'{pct:.0f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_missing_values.png', bbox_inches='tight')
plt.show()
print('\nKey finding: OMIM identifiers are missing for ~35% of cases, limiting gene-disease linkage analysis.')

## 3. Disease Distribution: How Rare is "Rare"?

In [ ]:
disease_counts = df_train['Disease'].value_counts()

# Bin into rarity tiers
bins   = [0, 1, 5, 20, 100, disease_counts.max() + 1]
labels = ['Ultra-rare (1)', 'Very rare (2–5)', 'Rare (6–20)',
          'Moderate (21–100)', 'Common (100+)']
tier_counts = pd.cut(disease_counts, bins=bins, labels=labels, right=True).value_counts().reindex(labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: rarity tier bar chart
colors = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4']
axes[0].bar(tier_counts.index, tier_counts.values, color=colors)
axes[0].set_xlabel('Rarity Tier (cases per disease)')
axes[0].set_ylabel('Number of Diseases')
axes[0].set_title('Disease Rarity Tier Distribution\n(Training Set)')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontsize=10, fontweight='bold')

# Right: log-scale frequency plot (Zipf-like)
axes[1].plot(range(len(disease_counts)), disease_counts.values, color='#1f77b4', linewidth=1.5)
axes[1].set_yscale('log')
axes[1].set_xlabel('Disease rank')
axes[1].set_ylabel('Case count (log scale)')
axes[1].set_title('Disease Frequency Distribution\n(Zipf-like long tail)')
axes[1].axvline(100, color='red', linestyle='--', alpha=0.6, label='Top 100 diseases')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_disease_distribution.png', bbox_inches='tight')
plt.show()

print(f'Total unique diseases: {len(disease_counts):,}')
print(f'Top 10 diseases account for {disease_counts.head(10).sum() / len(df_train) * 100:.1f}% of cases')
print(f'Diseases with only 1 case: {(disease_counts == 1).sum()} ({(disease_counts == 1).mean()*100:.1f}% of diseases)')

## 4. Clinical Text Analysis: Case Summary Length

In [ ]:
df_train['text_len']  = df_train['CaseSummary'].str.len()
df_train['word_count'] = df_train['CaseSummary'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length distribution
axes[0].hist(df_train['text_len'].clip(upper=10000), bins=60,
             color='#4c72b0', edgecolor='white', alpha=0.85)
axes[0].axvline(df_train['text_len'].median(), color='red',
                linestyle='--', label=f'Median = {df_train["text_len"].median():,.0f} chars')
axes[0].set_xlabel('Character count (capped at 10k)')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('Case Summary Length Distribution')
axes[0].legend()

# Word count by rarity tier
bins2   = [0, 1, 5, 20, 100, disease_counts.max() + 1]
labels2 = ['Ultra-rare\n(1)', 'Very rare\n(2–5)', 'Rare\n(6–20)', 'Moderate\n(21–100)', 'Common\n(100+)']
df_train['tier'] = pd.cut(
    df_train['Disease'].map(disease_counts), bins=bins2, labels=labels2, right=True
)
df_train.boxplot(column='word_count', by='tier', ax=axes[1],
                 medianprops=dict(color='red', linewidth=2), patch_artist=True)
axes[1].set_ylim(0, 1500)
axes[1].set_title('Word Count by Disease Rarity Tier')
axes[1].set_xlabel('Rarity Tier')
axes[1].set_ylabel('Word count')
plt.suptitle('')

plt.tight_layout()
plt.savefig('fig_text_length.png', bbox_inches='tight')
plt.show()

## 5. ICD-10 Chapter Analysis: Disease Domain Coverage

In [ ]:
# ICD-10 chapter letter → clinical domain mapping
ICD_CHAPTERS = {
    'A': 'Infectious/Parasitic', 'B': 'Infectious/Parasitic',
    'C': 'Neoplasms',
    'D': 'Blood/Neoplasms',
    'E': 'Endocrine/Metabolic',
    'F': 'Mental Disorders',
    'G': 'Nervous System',
    'H': 'Eye/Ear',
    'I': 'Circulatory',
    'J': 'Respiratory',
    'K': 'Digestive',
    'L': 'Skin',
    'M': 'Musculoskeletal',
    'N': 'Genitourinary',
    'O': 'Pregnancy',
    'P': 'Perinatal',
    'Q': 'Congenital/Chromosomal',
    'R': 'Symptoms/Signs',
    'S': 'Injury/Poisoning',
    'T': 'Injury/Poisoning',
    'Z': 'Health Status'
}

df_train['ICD_chapter_letter'] = df_train['ICD-10'].str[0].str.upper()
df_train['ICD_domain'] = df_train['ICD_chapter_letter'].map(ICD_CHAPTERS).fillna('Other')

domain_counts = df_train['ICD_domain'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
palette  = sns.color_palette('tab10', len(domain_counts))
bars = ax.barh(domain_counts.index[::-1], domain_counts.values[::-1], color=palette[::-1])
ax.set_xlabel('Number of cases')
ax.set_title('Case Distribution by Clinical Domain (ICD-10 Chapter)')
for bar, val in zip(bars, domain_counts.values[::-1]):
    ax.text(val + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_icd_chapters.png', bbox_inches='tight')
plt.show()

## 6. Train / Eval Split Alignment

In [ ]:
train_diseases = set(df_train['Disease'].unique())
eval_diseases  = set(df_eval['Disease'].unique())

overlap    = train_diseases & eval_diseases
eval_only  = eval_diseases - train_diseases
train_only = train_diseases - eval_diseases

print('=== Train / Eval Disease Overlap ===')
print(f'  Diseases in BOTH train & eval : {len(overlap):,} ({len(overlap)/len(eval_diseases)*100:.1f}% of eval)')
print(f'  Diseases in eval ONLY (novel) : {len(eval_only):,}')
print(f'  Diseases in train ONLY        : {len(train_only):,}')
print()
print('⚠️  Research implication: LLMs must generalise to', len(eval_only),
      'diseases NEVER seen in training context.')

# Pie chart
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    [len(overlap), len(eval_only)],
    labels=[f'Seen in training\n({len(overlap)} diseases)',
            f'Novel to eval\n({len(eval_only)} diseases)'],
    colors=['#4c72b0', '#dd8452'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax.set_title('Evaluation Disease Overlap with Training Set')
plt.savefig('fig_train_eval_overlap.png', bbox_inches='tight')
plt.show()

## 7. Ontology Coverage Analysis

Each case has links to multiple medical ontologies (OMIM, GARD, UMLS, ICD-10, ICD-11). Understanding coverage is crucial for any knowledge-grounding approach.

In [ ]:
ontology_cols = ['ICD-10', 'ICD-11', 'OMIM', 'GARD', 'UMLS']

# Coverage per case (how many ontologies are filled)
df_train['ontology_coverage'] = df_train[ontology_cols].notna().sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Coverage count histogram
cov_counts = df_train['ontology_coverage'].value_counts().sort_index()
axes[0].bar(cov_counts.index, cov_counts.values,
            color=sns.color_palette('Blues_r', len(cov_counts)))
axes[0].set_xlabel('Number of ontologies covered (out of 5)')
axes[0].set_ylabel('Cases')
axes[0].set_title('Ontology Coverage per Case')
for i, v in zip(cov_counts.index, cov_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

# Heatmap of per-ontology coverage by ICD domain
domain_cov = df_train.groupby('ICD_domain')[ontology_cols].apply(
    lambda g: g.notna().mean() * 100
).round(1)
sns.heatmap(domain_cov, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Coverage %'})
axes[1].set_title('Ontology Coverage % by Clinical Domain')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('fig_ontology_coverage.png', bbox_inches='tight')
plt.show()

## 8. Key Findings & Next Steps

### Summary

| Dimension | Finding | Research Implication |
|-----------|---------|---------------------|
| **Scale** | 50k train / 6.9k eval, 1,393 / 1,012 unique diseases | Rich enough for meaningful benchmarking |
| **Class imbalance** | Top 10 diseases = ~10% of cases; 158 diseases have only 1 case | LLMs likely underperform on ultra-rare classes |
| **Novel diseases in eval** | ~13% of eval diseases not seen in training context | Tests true zero-shot generalization |
| **Text length** | Median ~1,864 chars; range 1–65k | Long cases may exceed LLM context limits |
| **Ontology gaps** | OMIM missing ~35%, GARD ~21% | Limits gene-disease grounding approaches |
| **Domain skew** | Congenital (Q) and Neoplasms (C) dominate | LLMs may be systematically weaker on metabolic/neurological rare diseases |

### Recommended Next Analyses

1. **LLM Benchmarking** — Feed case summaries to GPT-4, Claude, Llama; measure top-1 / top-5 accuracy  
2. **Error Pattern Analysis** — Which disease categories / rarity tiers fail most?  
3. **Prompt Sensitivity Study** — Does rephrasing the summary change LLM diagnosis?  
4. **Confidence Calibration** — Are LLMs overconfident on incorrect ultra-rare diagnoses?  
5. **Text Length Effect** — Does truncating long summaries hurt performance?